# LeetCode #209 — Minimum Size Subarray Sum: cửa sổ trượt co giãn

**Mức độ:** Medium · **Chủ đề:** Array, Binary Search, Sliding Window, Prefix Sum

Skeleton trên LeetCode:

```python
class Solution:
    def minSubArrayLen(self, target: int, nums: List[int]) -> int:
```

Mục tiêu của notebook:

1. Baseline `O(n²)` duyệt mọi điểm đầu.
2. Nhận ra vì sao **mọi số dương** cho phép dùng **cửa sổ trượt**: mở rộng phải đến khi đủ, rồi **co trái** càng nhiều càng tốt.
3. Chứng minh `O(n)`: mỗi chỉ số vào cửa sổ một lần, ra một lần.
4. Cách thứ hai `O(n log n)`: **tổng tiền tố** tăng dần + **tìm kiếm nhị phân** (follow-up của đề).

# 1. Bài toán là gì?

Cho mảng số **nguyên dương** `nums` và số nguyên dương `target`. Tìm **độ dài nhỏ nhất** của một mảng con liên tiếp có tổng **`>= target`**. Nếu không có, trả về `0`.

**Constraints (theo đề):**

- `1 <= target <= 10^9`
- `1 <= nums.length <= 10^5`
- `1 <= nums[i] <= 10^4`

Follow-up: nếu đã có lời giải `O(n)`, hãy thử thêm lời giải `O(n log n)`.

---

## Ví dụ

```text
target = 7,  nums = [2, 3, 1, 2, 4, 3]       ->  2   ([4, 3])
target = 4,  nums = [1, 4, 4]                ->  1   ([4])
target = 11, nums = [1, 1, 1, 1, 1, 1, 1, 1] ->  0   (tổng cả mảng chỉ là 8)
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Cửa sổ trượt (sliding window)
2. Tổng tiền tố (prefix sum) và tổng chạy
3. Tìm kiếm nhị phân (binary search)

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Cửa sổ trượt (sliding window)

### Ý tưởng

Cửa sổ trượt là trường hợp riêng của hai con trỏ cùng chiều: hai đầu `left`, `right` xác định một **đoạn liên tiếp** `[left, right]`. Ta **mở rộng** bên phải từng bước; khi cửa sổ vi phạm điều kiện, **thu hẹp** bên trái cho tới khi hợp lệ lại. Mỗi phần tử vào cửa sổ một lần, ra một lần → `O(n)`.

```text
Đoạn con dài nhất có tổng <= 7 trong [2, 1, 5, 1, 3, 2]
[2]           tổng 2   ✓  dài 1
[2,1]         tổng 3   ✓  dài 2
[2,1,5]       tổng 8   ✗  → bỏ 2 → [1,5] tổng 6 ✓
[1,5,1]       tổng 7   ✓  dài 3
[1,5,1,3]     tổng 10  ✗  → bỏ 1, bỏ 5 → [1,3] tổng 4
[1,3,2]       tổng 6   ✓  dài 3
```

### Khung (cửa sổ thay đổi kích thước)

```text
left = 0
for right in range(n):
    thêm a[right] vào cửa sổ
    while cửa sổ không hợp lệ:
        bỏ a[left] khỏi cửa sổ; left += 1
    cập nhật đáp án với cửa sổ [left, right]
```

In [1]:
def dai_nhat_tong_khong_qua(a, k):
    left, tong, best = 0, 0, 0
    for right, x in enumerate(a):
        tong += x
        while tong > k:
            tong -= a[left]
            left += 1
        best = max(best, right - left + 1)
        print(f"   right={right} cửa sổ={a[left:right + 1]} tổng={tong}")
    return best

print("→ dài nhất:", dai_nhat_tong_khong_qua([2, 1, 5, 1, 3, 2], 7))

   right=0 cửa sổ=[2] tổng=2
   right=1 cửa sổ=[2, 1] tổng=3
   right=2 cửa sổ=[1, 5] tổng=6
   right=3 cửa sổ=[1, 5, 1] tổng=7
   right=4 cửa sổ=[1, 3] tổng=4
   right=5 cửa sổ=[1, 3, 2] tổng=6
→ dài nhất: 3


### Cửa sổ với bảng băm: đoạn không có ký tự lặp

Cửa sổ giữ **tập ký tự** bên trong. Khi ký tự mới đã có trong cửa sổ, thu hẹp trái cho tới khi bản cũ bị bỏ ra.

In [2]:
def dai_nhat_khong_lap(s):
    trong, left, best = set(), 0, 0
    for right, ch in enumerate(s):
        while ch in trong:
            trong.remove(s[left])
            left += 1
        trong.add(ch)
        best = max(best, right - left + 1)
    return best

for s_sw in ["abcabcbb", "bbbbb", "pwwkew", ""]:
    print(f"{s_sw!r:>11}: {dai_nhat_khong_lap(s_sw)}")

 'abcabcbb': 3
    'bbbbb': 1
   'pwwkew': 3
         '': 0


### Lỗi hay gặp

- Dùng `if` thay cho `while` khi thu hẹp → cửa sổ có thể vẫn chưa hợp lệ.
- Cập nhật đáp án **trước** khi làm cửa sổ hợp lệ.
- Áp dụng cho bài mà thu hẹp không giữ tính đơn điệu (ví dụ tổng với số **âm**: bỏ bớt phần tử chưa chắc làm tổng giảm).

## 2.2 Tổng tiền tố (prefix sum) và tổng chạy

### Tổng tiền tố là gì?

Với mảng `a`, mảng tổng tiền tố `P` có `P[0] = 0` và `P[i] = a[0] + … + a[i−1]`. Khi đó tổng một đoạn bất kỳ:

```text
a[l] + … + a[r] = P[r + 1] − P[l]          (O(1) sau khi dựng P trong O(n))
```

In [3]:
from itertools import accumulate

a_ps = [3, -1, 4, 1, -5, 9]
P_ps = [0] + list(accumulate(a_ps))
print("a =", a_ps)
print("P =", P_ps)
l_ps, r_ps = 1, 3
print(f"tổng a[{l_ps}..{r_ps}] = P[{r_ps + 1}] - P[{l_ps}] = {P_ps[r_ps + 1] - P_ps[l_ps]} =", sum(a_ps[l_ps:r_ps + 1]))

a = [3, -1, 4, 1, -5, 9]
P = [0, 3, 2, 6, 7, 2, 11]
tổng a[1..3] = P[4] - P[1] = 4 = 4


### Tổng chạy và điểm "thấp nhất"

Trong nhiều bài, ta chỉ cần **tổng chạy** (running sum) vừa duyệt vừa cộng. Một quan sát hay dùng cho mảng **vòng tròn** có tổng toàn bộ `≥ 0`: nếu tổng chạy **xuống thấp nhất** tại vị trí `k`, thì xuất phát ngay sau `k` (đi vòng) mọi tổng chạy đều `≥ 0` — không có đoạn nào làm tổng bị âm.

In [4]:
b_ps = [1, -3, 2, -1, 3, -1]                 # tổng = 1 >= 0
tong_chay, thap_nhat, vi_tri = 0, 0, -1
for i, v in enumerate(b_ps):
    tong_chay += v
    if tong_chay < thap_nhat:
        thap_nhat, vi_tri = tong_chay, i
    print(f"   i={i} a[i]={v:>2} tổng chạy={tong_chay:>2}")
print("tổng chạy thấp nhất", thap_nhat, "ngay sau vị trí", vi_tri, "→ xuất phát từ", vi_tri + 1)
start_ps = vi_tri + 1
quay_vong = b_ps[start_ps:] + b_ps[:start_ps]
print("xuất phát từ 0 thì các tổng chạy:", list(accumulate(b_ps)), "(có số âm)")
print("xuất phát từ đó, các tổng chạy:", list(accumulate(quay_vong)))

   i=0 a[i]= 1 tổng chạy= 1
   i=1 a[i]=-3 tổng chạy=-2
   i=2 a[i]= 2 tổng chạy= 0
   i=3 a[i]=-1 tổng chạy=-1
   i=4 a[i]= 3 tổng chạy= 2
   i=5 a[i]=-1 tổng chạy= 1
tổng chạy thấp nhất -2 ngay sau vị trí 1 → xuất phát từ 2
xuất phát từ 0 thì các tổng chạy: [1, -2, 0, -1, 2, 1] (có số âm)
xuất phát từ đó, các tổng chạy: [2, 1, 4, 3, 4, 1]


Lỗi hay gặp: lệch chỉ số giữa `P[i]` và `a[i]`; quên `P[0] = 0`; dùng `sum(a[l:r+1])` trong vòng lặp (thành `O(n²)`).

## 2.3 Tìm kiếm nhị phân (binary search)

### Ý tưởng

Trên mảng **đã sắp xếp**, so phần tử cần tìm với phần tử **ở giữa**: nếu nhỏ hơn thì chắc chắn nằm ở nửa trái, lớn hơn thì ở nửa phải. Mỗi bước loại **một nửa** → chỉ `O(log n)` bước (mảng 1 triệu phần tử: ~20 bước).

```text
tìm 23 trong [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
lo=0 hi=9 mid=4 → 16 < 23 → lo = 5
lo=5 hi=9 mid=7 → 56 > 23 → hi = 6
lo=5 hi=6 mid=5 → 23 = 23 → tìm thấy ở chỉ số 5
```

In [5]:
def tim_nhi_phan(a, x):
    lo, hi = 0, len(a) - 1                # khoảng tìm đóng [lo, hi]
    while lo <= hi:
        mid = (lo + hi) // 2
        print(f"   lo={lo} hi={hi} mid={mid} a[mid]={a[mid]}")
        if a[mid] == x:
            return mid
        if a[mid] < x:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

a_bs = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
print("→", tim_nhi_phan(a_bs, 23))
print("→", tim_nhi_phan(a_bs, 7), "(không có)")

   lo=0 hi=9 mid=4 a[mid]=16
   lo=5 hi=9 mid=7 a[mid]=56
   lo=5 hi=6 mid=5 a[mid]=23
→ 5
   lo=0 hi=9 mid=4 a[mid]=16
   lo=0 hi=3 mid=1 a[mid]=5
   lo=2 hi=3 mid=2 a[mid]=8
→ -1 (không có)


### Dạng tổng quát: tìm vị trí **đầu tiên** thỏa điều kiện

Rất nhiều bài không hỏi "có `x` không" mà hỏi "vị trí đầu tiên mà điều kiện `P(i)` đúng", với `P` **đơn điệu**: `False False … False True True … True`. Khung:

```text
lo, hi = 0, n            # đáp án nằm trong [lo, hi]; n nghĩa là "không có"
while lo < hi:
    mid = (lo + hi) // 2
    if P(mid): hi = mid          # mid có thể là đáp án → giữ lại
    else:      lo = mid + 1      # mid chắc chắn không phải
return lo
```

Hai trường hợp quan trọng:

- **lower_bound**: vị trí đầu tiên `a[i] >= x` (`bisect_left`).
- **upper_bound**: vị trí đầu tiên `a[i] > x` (`bisect_right`).

Số lần xuất hiện của `x` = `upper_bound − lower_bound`.

In [6]:
from bisect import bisect_left, bisect_right

def dau_tien(n, P):
    lo, hi = 0, n
    while lo < hi:
        mid = (lo + hi) // 2
        if P(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

b_bs = [1, 2, 2, 2, 5, 7]
lb = dau_tien(len(b_bs), lambda i: b_bs[i] >= 2)
ub = dau_tien(len(b_bs), lambda i: b_bs[i] > 2)
print("lower_bound(2) =", lb, "| upper_bound(2) =", ub, "| số lần xuất hiện:", ub - lb)
print("bisect_left / bisect_right:", bisect_left(b_bs, 2), bisect_right(b_bs, 2))
print("chèn 6 vào vị trí:", bisect_left(b_bs, 6))
print("số nguyên nhỏ nhất k với k*k >= 50:", dau_tien(100, lambda k: k * k >= 50))

lower_bound(2) = 1 | upper_bound(2) = 4 | số lần xuất hiện: 3
bisect_left / bisect_right: 1 4
chèn 6 vào vị trí: 5
số nguyên nhỏ nhất k với k*k >= 50: 8


### Lỗi hay gặp (nguồn lỗi số 1 là "lệch 1")

- Trộn hai kiểu khoảng: `while lo <= hi` đi với `hi = mid - 1`; `while lo < hi` đi với `hi = mid`. Trộn lẫn → **lặp vô hạn** hoặc bỏ sót.
- `lo = mid` (thay vì `mid + 1`) với `mid = (lo+hi)//2` → khi `hi = lo + 1` thì `mid = lo`, không tiến → lặp vô hạn.
- Áp dụng trên mảng **chưa sắp xếp** / điều kiện không đơn điệu.
- (Ngôn ngữ khác) `(lo + hi) / 2` tràn số → dùng `lo + (hi - lo) // 2`; Python không bị.

In [7]:
def sai_lo_bang_mid(a, x, gioi_han=20):
    lo, hi, buoc = 0, len(a) - 1, 0
    while lo < hi and buoc < gioi_han:
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid            # SAI: không loại mid
        else:
            hi = mid
        buoc += 1
    return buoc
print("lo = mid: số vòng trước khi bị chặn:", sai_lo_bang_mid([1, 3], 3))

lo = mid: số vòng trước khi bị chặn: 20


## 2.4 Áp dụng vào bài này

Mọi số **dương** nên mở rộng cửa sổ làm tổng tăng, co lại làm tổng giảm → **cửa sổ trượt** co giãn: thêm phần tử bên phải, rồi `while` tổng `>= target` thì ghi nhận độ dài và bỏ phần tử bên trái — `O(n)`. Follow-up `O(n log n)`: **tổng tiền tố** tăng nghiêm ngặt, với mỗi điểm đầu **tìm nhị phân** điểm cuối nhỏ nhất đủ tổng.

# 3. Python cơ bản cần biết

In [8]:
from typing import List
from itertools import accumulate
from bisect import bisect_left
import random, time

nums = [2, 3, 1, 2, 4, 3]
print("tổng tiền tố (có 0 ở đầu):", list(accumulate(nums, initial=0)))
print("float('inf') làm giá trị khởi tạo cho min:", min(float("inf"), 5))
print("bisect_left([0, 2, 5, 6, 8], 5) =", bisect_left([0, 2, 5, 6, 8], 5), "(chỉ số đầu tiên có giá trị >= 5)")

tổng tiền tố (có 0 ở đầu): [0, 2, 5, 6, 8, 12, 15]
float('inf') làm giá trị khởi tạo cho min: 5
bisect_left([0, 2, 5, 6, 8], 5) = 2 (chỉ số đầu tiên có giá trị >= 5)


# 4. Baseline: thử mọi điểm đầu

Với mỗi `i`, cộng dần sang phải đến khi tổng `>= target` → độ dài nhỏ nhất bắt đầu tại `i`. `O(n²)`.

In [9]:
def min_len_brute(target: int, nums: List[int]) -> int:
    best = float("inf")
    for i in range(len(nums)):
        s = 0
        for j in range(i, len(nums)):
            s += nums[j]
            if s >= target:
                best = min(best, j - i + 1)
                break                      # kéo dài thêm chỉ làm dài hơn
    return 0 if best == float("inf") else best

print(min_len_brute(7, [2, 3, 1, 2, 4, 3]), min_len_brute(4, [1, 4, 4]), min_len_brute(11, [1] * 8))

2

 1 0


# 5. Ý tưởng chính: cửa sổ trượt

Vì **mọi phần tử dương**:

- Mở rộng cửa sổ sang phải → tổng **tăng**.
- Thu hẹp từ bên trái → tổng **giảm**.

Hệ quả: nếu `[left, right]` đã đủ (`>= target`) thì mọi cửa sổ `[left, right']` với `right' > right` đều đủ nhưng **dài hơn** — vô ích. Và khi chuyển sang `right + 1`, điểm đầu tốt nhất **không thể lùi về trái** — `left` chỉ tiến.

```text
left = 0, s = 0
for right in range(n):
    s += nums[right]                    # mở rộng
    while s >= target:                  # đủ rồi → ghi nhận, rồi co trái
        best = min(best, right - left + 1)
        s -= nums[left]; left += 1
```

In [10]:
def min_len_window(target: int, nums: List[int]) -> int:
    left = s = 0
    best = float("inf")
    for right, x in enumerate(nums):
        s += x
        while s >= target:
            best = min(best, right - left + 1)
            s -= nums[left]
            left += 1
    return 0 if best == float("inf") else best

print(min_len_window(7, [2, 3, 1, 2, 4, 3]), min_len_window(4, [1, 4, 4]), min_len_window(11, [1] * 8))

2 1 0


# 6. Mô phỏng

In [11]:
def trace_window(target, nums):
    left = s = 0
    best = float("inf")
    for right, x in enumerate(nums):
        s += x
        print(f"thêm nums[{right}]={x}: cửa sổ {nums[left:right + 1]} tổng {s}")
        while s >= target:
            best = min(best, right - left + 1)
            print(f"   đủ (>= {target}) → ghi nhận độ dài {right - left + 1}, best = {best}; bỏ nums[{left}]={nums[left]}")
            s -= nums[left]
            left += 1
    print("kết quả:", 0 if best == float("inf") else best)

trace_window(7, [2, 3, 1, 2, 4, 3])

thêm nums[0]=2: cửa sổ [2] tổng 2


thêm nums[1]=3: cửa sổ [2, 3] tổng 5
thêm nums[2]=1: cửa sổ [2, 3, 1] tổng 6
thêm nums[3]=2: cửa sổ [2, 3, 1, 2] tổng 8
   đủ (>= 7) → ghi nhận độ dài 4, best = 4; bỏ nums[0]=2
thêm nums[4]=4: cửa sổ [3, 1, 2, 4] tổng 10
   đủ (>= 7) → ghi nhận độ dài 4, best = 4; bỏ nums[1]=3
   đủ (>= 7) → ghi nhận độ dài 3, best = 3; bỏ nums[2]=1
thêm nums[5]=3: cửa sổ [2, 4, 3] tổng 9
   đủ (>= 7) → ghi nhận độ dài 3, best = 3; bỏ nums[3]=2
   đủ (>= 7) → ghi nhận độ dài 2, best = 2; bỏ nums[4]=4
kết quả: 2


# 7. Cách 2: tổng tiền tố + tìm kiếm nhị phân `O(n log n)`

Gọi `P[k] = nums[0] + ... + nums[k-1]` (`P[0] = 0`). Tổng đoạn `nums[i..j-1] = P[j] − P[i]`.

Vì mọi số dương nên `P` **tăng nghiêm ngặt**. Với mỗi điểm đầu `i`, cần `j` nhỏ nhất sao cho `P[j] >= P[i] + target` → **`bisect_left(P, P[i] + target)`**.

In [12]:
def min_len_bisect(target: int, nums: List[int]) -> int:
    P = list(accumulate(nums, initial=0))
    n = len(nums)
    best = float("inf")
    for i in range(n):
        j = bisect_left(P, P[i] + target)
        if j <= n:
            best = min(best, j - i)
    return 0 if best == float("inf") else best

P = list(accumulate([2, 3, 1, 2, 4, 3], initial=0))
print("P =", P)
for i in range(6):
    j = bisect_left(P, P[i] + 7)
    print(f"   i={i}: cần P[j] >= {P[i] + 7} → j = {j}" + (f", độ dài {j - i}" if j <= 6 else " (không có)"))
print("kết quả:", min_len_bisect(7, [2, 3, 1, 2, 4, 3]))

P = [0, 2, 5, 6, 8, 12, 15]
   i=0: cần P[j] >= 7 → j = 4, độ dài 4
   i=1: cần P[j] >= 9 → j = 5, độ dài 4
   i=2: cần P[j] >= 12 → j = 5, độ dài 3
   i=3: cần P[j] >= 13 → j = 6, độ dài 3
   i=4: cần P[j] >= 15 → j = 6, độ dài 2
   i=5: cần P[j] >= 19 → j = 7 (không có)
kết quả: 2


# 8. Phiên bản đúng format LeetCode

In [13]:
class Solution:
    def minSubArrayLen(self, target: int, nums: List[int]) -> int:
        left = s = 0
        best = len(nums) + 1                 # "vô cực" kiểu số nguyên
        for right, x in enumerate(nums):
            s += x
            while s >= target:
                if right - left + 1 < best:
                    best = right - left + 1
                s -= nums[left]
                left += 1
        return 0 if best == len(nums) + 1 else best

In [14]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<16} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.minSubArrayLen(7, [2, 3, 1, 2, 4, 3]), 2)
check("Ví dụ 2", sol.minSubArrayLen(4, [1, 4, 4]), 1)
check("Ví dụ 3", sol.minSubArrayLen(11, [1] * 8), 0)
check("Cả mảng", sol.minSubArrayLen(15, [1, 2, 3, 4, 5]), 5)
check("Một phần tử", sol.minSubArrayLen(3, [3]), 1)

random.seed(209)
for _ in range(3000):
    nums = [random.randint(1, 10) for _ in range(random.randint(1, 12))]
    target = random.randint(1, 60)
    exp = min_len_brute(target, nums)
    assert sol.minSubArrayLen(target, nums) == exp == min_len_bisect(target, nums), (target, nums)
print("3000 test ngẫu nhiên: cửa sổ trượt và nhị phân khớp brute force")

big = [random.randint(1, 10**4) for _ in range(10**5)]
T = sum(big) // 3
for name, f in [("cửa sổ O(n)", sol.minSubArrayLen), ("nhị phân O(n log n)", min_len_bisect)]:
    t0 = time.time(); r = f(T, big); t1 = time.time()
    print(f"{name:<20}: {r}, {t1 - t0:.3f}s")
small_nums = big[:3000]
t0 = time.time(); r = min_len_brute(sum(small_nums) // 3, small_nums); t1 = time.time()
print(f"brute O(n²), chỉ n=3000: {r}, {t1 - t0:.3f}s")

Ví dụ 1          PASS  2
Ví dụ 2          PASS  1
Ví dụ 3          PASS  0
Cả mảng          PASS  5
Một phần tử      PASS  1


3000 test ngẫu nhiên: cửa sổ trượt và nhị phân khớp brute force


cửa sổ O(n)         : 33243, 0.015s
nhị phân O(n log n) : 33243, 0.026s


brute O(n²), chỉ n=3000: 976, 0.121s


# 9. Vì sao đúng? Độ phức tạp?

**Bất biến:** khi vòng `while` kết thúc ở bước `right`, `[left, right]` là cửa sổ kết thúc tại `right` có tổng `< target`, và mọi điểm đầu `< left` đã được xét (cửa sổ đủ ngắn nhất kết thúc tại một `right'` nào đó đã được ghi nhận khi `left` đi qua).

Cụ thể: với mỗi `right`, đoạn ngắn nhất kết thúc tại `right` bắt đầu ở `L(right)` = chỉ số lớn nhất mà tổng `>= target`. Vì số dương, `L(right)` **không giảm** khi `right` tăng → `left` chỉ cần tiến, và vòng `while` dừng đúng sau khi ghi nhận `L(right)`.

```text
Cửa sổ trượt:  Time O(n)  — mỗi chỉ số vào 1 lần (right), ra tối đa 1 lần (left)
               Space O(1)
Nhị phân:      Time O(n log n), Space O(n) cho mảng tiền tố
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Dùng `if` thay cho `while` khi co trái

In [15]:
def wrong_if(target, nums):
    left = s = 0
    best = float("inf")
    for right, x in enumerate(nums):
        s += x
        if s >= target:                      # SAI: chỉ co được 1 bước mỗi lần
            best = min(best, right - left + 1)
            s -= nums[left]; left += 1
    return 0 if best == float("inf") else best

print("target=100, [1,1,1,1,100] — sai:", wrong_if(100, [1, 1, 1, 1, 100]), "| đúng:", sol.minSubArrayLen(100, [1, 1, 1, 1, 100]))

target=100, [1,1,1,1,100] — sai: 5 | đúng: 1


## Lỗi 2: Điều kiện `>` thay vì `>=`

In [16]:
def wrong_strict(target, nums):
    left = s = 0
    best = float("inf")
    for right, x in enumerate(nums):
        s += x
        while s > target:                     # SAI
            best = min(best, right - left + 1)
            s -= nums[left]; left += 1
    return 0 if best == float("inf") else best

print("Ví dụ 2 — sai:", wrong_strict(4, [1, 4, 4]), "| đúng:", sol.minSubArrayLen(4, [1, 4, 4]))

Ví dụ 2 — sai: 2 | đúng: 1


## Lỗi 3: Quên trường hợp không có đáp án

Trả về `best` trực tiếp → `inf` (hoặc `n + 1`) thay vì `0`.

In [17]:
def wrong_no_zero(target, nums):
    left = s = 0
    best = float("inf")
    for right, x in enumerate(nums):
        s += x
        while s >= target:
            best = min(best, right - left + 1)
            s -= nums[left]; left += 1
    return best                               # SAI

print("Ví dụ 3 — sai:", wrong_no_zero(11, [1] * 8), "| đúng:", sol.minSubArrayLen(11, [1] * 8))

Ví dụ 3 — sai: inf | đúng: 0


## Lỗi 4: Áp dụng cửa sổ trượt khi có **số âm**

Nếu cho phép số âm, mở rộng cửa sổ có thể làm tổng **giảm** → lập luận "left chỉ tiến" sụp đổ.

In [18]:
random.seed(4)
for _ in range(10000):
    nums = [random.randint(-5, 10) for _ in range(random.randint(1, 6))]
    target = random.randint(1, 15)
    if min_len_window(target, nums) != min_len_brute(target, nums):
        print("Có số âm:", nums, "target =", target, "→ cửa sổ:", min_len_window(target, nums), "| brute:", min_len_brute(target, nums))
        break

Có số âm: [4, -4, 2, 6] target = 5 → cửa sổ: 4 | brute: 1


Với số âm, bài trở thành **#862 Shortest Subarray with Sum at Least K** (Hard) — cần hàng đợi đơn điệu trên tổng tiền tố. Lưu ý `min_len_brute` ở trên dùng `break` sớm, vẫn đúng khi có số âm vì nó dừng ở `j` **đầu tiên** đạt đủ với điểm đầu `i` — đoạn ngắn nhất bắt đầu tại `i`.

# 11. Mẹo nhận diện pattern

- **"Mảng con liên tiếp ngắn/dài nhất thỏa điều kiện đơn điệu"** (mở rộng làm điều kiện dễ đạt hơn) → cửa sổ trượt hai con trỏ.
- **Mọi số không âm + tổng đoạn** → tổng tiền tố **đơn điệu** → có thể tìm nhị phân.
- **Có số âm** → tổng tiền tố + hàng đợi đơn điệu (#862) hoặc hash map (#560).

Bài liên quan: **#3 Longest Substring Without Repeating Characters**, **#713 Subarray Product Less Than K**, **#1004 Max Consecutive Ones III**, **#862 Shortest Subarray with Sum at Least K** (Hard), **#76 Minimum Window Substring** (Hard), **#560 Subarray Sum Equals K**.

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với `target = 15`, `nums = [5, 1, 3, 5, 10, 7, 4, 9, 2, 8]`, chạy cửa sổ trượt và ghi lại mọi độ dài được ghi nhận.

## Bài 2 — Điền chỗ trống

In [19]:
def exercise_2_min_sub_array_len(target: int, nums: List[int]) -> int:
    left = s = 0
    best = float("inf")
    for right, x in enumerate(nums):
        s += x
        # TODO: co trái khi đủ
        pass
    return 0 if best == float("inf") else best


# assert exercise_2_min_sub_array_len(7, [2, 3, 1, 2, 4, 3]) == 2
# assert exercise_2_min_sub_array_len(4, [1, 4, 4]) == 1
# assert exercise_2_min_sub_array_len(11, [1, 1, 1, 1, 1, 1, 1, 1]) == 0

## Bài 3 — Biến thể: mảng con **dài nhất** có tổng `<= k` (mọi số dương)

In [20]:
def exercise_3_longest_at_most(k: int, nums: List[int]) -> int:
    # TODO
    pass


# assert exercise_3_longest_at_most(5, [1, 2, 1, 1, 3]) == 4
# assert exercise_3_longest_at_most(1, [2, 3]) == 0
# assert exercise_3_longest_at_most(100, [5, 5, 5]) == 3

## Bài 4 — Biến thể: Subarray Product Less Than K (LeetCode #713)

Đếm số mảng con liên tiếp có **tích nghiêm ngặt nhỏ hơn** `k` (mọi số dương). Gợi ý: với mỗi `right`, số mảng con hợp lệ kết thúc tại `right` là `right − left + 1`.

In [21]:
def exercise_4_num_subarray_product_less_than_k(nums: List[int], k: int) -> int:
    # TODO
    pass


# assert exercise_4_num_subarray_product_less_than_k([10, 5, 2, 6], 100) == 8
# assert exercise_4_num_subarray_product_less_than_k([1, 2, 3], 0) == 0
# assert exercise_4_num_subarray_product_less_than_k([1, 1, 1], 2) == 6

## Bài 5 — Mini challenge: Shortest Subarray with Sum at Least K (LeetCode #862, Hard)

Giống bài này nhưng `nums` **có thể âm**. Gợi ý: tổng tiền tố `P`; duyệt `j`, giữ deque các chỉ số `i` với `P[i]` **tăng dần**; khi `P[j] − P[deque[0]] >= k` thì ghi nhận và pop trái; trước khi thêm `j`, pop phải các `i` có `P[i] >= P[j]`.

In [22]:
def exercise_5_shortest_subarray(nums: List[int], k: int) -> int:
    # TODO: trả về -1 nếu không có (theo đề #862)
    pass


# assert exercise_5_shortest_subarray([1], 1) == 1
# assert exercise_5_shortest_subarray([1, 2], 4) == -1
# assert exercise_5_shortest_subarray([2, -1, 2], 3) == 3
# assert exercise_5_shortest_subarray([84, -37, 32, 40, 95], 167) == 3

# 13. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_window(15, ...)` sau khi tự làm.

**Bài 3**
- Mức 1: mở rộng phải; **khi tổng > k** thì co trái; sau đó cửa sổ hợp lệ → cập nhật độ dài **lớn nhất**.

**Bài 4**
- Mức 1: nếu `k <= 1` trả 0 ngay (tích số dương luôn `>= 1`).
- Mức 2: `prod *= x`; `while prod >= k: prod //= nums[left]; left += 1`; cộng `right − left + 1`.

**Bài 5**
- Mức 1: deque giữ "điểm đầu ứng viên". Một `i` bị pop phải khi có `j` sau nó với `P[j] <= P[i]` vì `j` vừa gần hơn vừa cho tổng lớn hơn.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [23]:
trace_window(15, [5, 1, 3, 5, 10, 7, 4, 9, 2, 8])

thêm nums[0]=5: cửa sổ [5] tổng 5
thêm nums[1]=1: cửa sổ [5, 1] tổng 6
thêm nums[2]=3: cửa sổ [5, 1, 3] tổng 9
thêm nums[3]=5: cửa sổ [5, 1, 3, 5] tổng 14
thêm nums[4]=10: cửa sổ [5, 1, 3, 5, 10] tổng 24
   đủ (>= 15) → ghi nhận độ dài 5, best = 5; bỏ nums[0]=5
   đủ (>= 15) → ghi nhận độ dài 4, best = 4; bỏ nums[1]=1
   đủ (>= 15) → ghi nhận độ dài 3, best = 3; bỏ nums[2]=3
   đủ (>= 15) → ghi nhận độ dài 2, best = 2; bỏ nums[3]=5
thêm nums[5]=7: cửa sổ [10, 7] tổng 17
   đủ (>= 15) → ghi nhận độ dài 2, best = 2; bỏ nums[4]=10
thêm nums[6]=4: cửa sổ [7, 4] tổng 11
thêm nums[7]=9: cửa sổ [7, 4, 9] tổng 20
   đủ (>= 15) → ghi nhận độ dài 3, best = 2; bỏ nums[5]=7
thêm nums[8]=2: cửa sổ [4, 9, 2] tổng 15
   đủ (>= 15) → ghi nhận độ dài 3, best = 2; bỏ nums[6]=4
thêm nums[9]=8: cửa sổ [9, 2, 8] tổng 19
   đủ (>= 15) → ghi nhận độ dài 3, best = 2; bỏ nums[7]=9
kết quả: 2


## Đáp án Bài 2

In [24]:
def answer_2_min_sub_array_len(target: int, nums: List[int]) -> int:
    left = s = 0
    best = float("inf")
    for right, x in enumerate(nums):
        s += x
        while s >= target:
            best = min(best, right - left + 1)
            s -= nums[left]
            left += 1
    return 0 if best == float("inf") else best

assert answer_2_min_sub_array_len(7, [2, 3, 1, 2, 4, 3]) == 2
assert answer_2_min_sub_array_len(4, [1, 4, 4]) == 1
assert answer_2_min_sub_array_len(11, [1, 1, 1, 1, 1, 1, 1, 1]) == 0
print("Bài 2 PASS")

Bài 2 PASS


## Đáp án Bài 3

In [25]:
def answer_3_longest_at_most(k: int, nums: List[int]) -> int:
    left = s = best = 0
    for right, x in enumerate(nums):
        s += x
        while s > k:
            s -= nums[left]
            left += 1
        best = max(best, right - left + 1)
    return best

def longest_brute(k, nums):
    n = len(nums)
    return max([j - i for i in range(n) for j in range(i + 1, n + 1) if sum(nums[i:j]) <= k], default=0)

assert answer_3_longest_at_most(5, [1, 2, 1, 1, 3]) == 4
assert answer_3_longest_at_most(1, [2, 3]) == 0
assert answer_3_longest_at_most(100, [5, 5, 5]) == 3
for _ in range(2000):
    nums = [random.randint(1, 10) for _ in range(random.randint(1, 10))]
    k = random.randint(1, 40)
    assert answer_3_longest_at_most(k, nums) == longest_brute(k, nums)
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [26]:
def answer_4_num_subarray_product_less_than_k(nums: List[int], k: int) -> int:
    if k <= 1:
        return 0
    prod, left, count = 1, 0, 0
    for right, x in enumerate(nums):
        prod *= x
        while prod >= k:
            prod //= nums[left]
            left += 1
        count += right - left + 1
    return count

def product_brute(nums, k):
    from math import prod
    n = len(nums)
    return sum(prod(nums[i:j]) < k for i in range(n) for j in range(i + 1, n + 1))

assert answer_4_num_subarray_product_less_than_k([10, 5, 2, 6], 100) == 8
assert answer_4_num_subarray_product_less_than_k([1, 2, 3], 0) == 0
assert answer_4_num_subarray_product_less_than_k([1, 1, 1], 2) == 6
for _ in range(2000):
    nums = [random.randint(1, 6) for _ in range(random.randint(1, 8))]
    k = random.randint(0, 60)
    assert answer_4_num_subarray_product_less_than_k(nums, k) == product_brute(nums, k)
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [27]:
from collections import deque

def answer_5_shortest_subarray(nums: List[int], k: int) -> int:
    P = list(accumulate(nums, initial=0))
    best = len(nums) + 1
    dq = deque()
    for j, pj in enumerate(P):
        while dq and pj - P[dq[0]] >= k:
            best = min(best, j - dq.popleft())
        while dq and P[dq[-1]] >= pj:
            dq.pop()
        dq.append(j)
    return best if best <= len(nums) else -1

assert answer_5_shortest_subarray([1], 1) == 1
assert answer_5_shortest_subarray([1, 2], 4) == -1
assert answer_5_shortest_subarray([2, -1, 2], 3) == 3
assert answer_5_shortest_subarray([84, -37, 32, 40, 95], 167) == 3
for _ in range(3000):
    nums = [random.randint(-5, 10) for _ in range(random.randint(1, 8))]
    k = random.randint(1, 20)
    b = min_len_brute(k, nums)
    assert answer_5_shortest_subarray(nums, k) == (b if b else -1), (nums, k)
print("Bài 5 PASS")

Bài 5 PASS


# 15. Tổng kết

> Mọi số dương ⟹ mở rộng làm tổng tăng, co lại làm tổng giảm ⟹ **cửa sổ trượt**: thêm `nums[right]`, rồi **`while`** tổng `>= target` thì ghi nhận độ dài và bỏ `nums[left]`. `O(n)` thời gian, `O(1)` bộ nhớ. Cách khác: tổng tiền tố tăng dần + `bisect_left` cho `O(n log n)`. Có số âm → cần deque đơn điệu (#862).

---

## Checklist trước khi nộp

- [ ] `while` (không phải `if`) khi co trái?
- [ ] Điều kiện `>= target`?
- [ ] Trả về `0` khi không có đáp án?
- [ ] Đã kiểm tra giả thiết mọi số dương?